In [11]:
# ==========================================
# 1. Mount Google Drive
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# ==========================================
# 2. Import Library & Setup Direktori
# ==========================================
import pandas as pd
import glob
import os

# Base path direktori penelitian
base_path = "/content/drive/My Drive/skripsi/dataset/mbg_2025/"

# Direktori pecahan file (batch)
seed_dir = os.path.join(base_path, "labeling_roberta/seed_batches_label_studio/*.csv")
refill_dir = os.path.join(base_path, "labeling_roberta/refill_batches_label_studio/*.csv")

# Path file Master
master_path = os.path.join(base_path, "merged_data/merged_mbg_full_with_text_duplication.csv")

# Path output file
output_dir = os.path.join(base_path, "processed/")
output_path = os.path.join(output_dir, "mbg_sampled_research.csv")

# Membuat folder output jika belum ada
os.makedirs(output_dir, exist_ok=True)

In [13]:
# ==========================================
# 3. Ekstraksi Data Batch Mentah (RAW)
# ==========================================
print("Mulai mengumpulkan data dari file pecahan...")

batch_dfs = []

def extract_data_from_files(file_pattern, source_name):
    files = glob.glob(file_pattern)
    print(f"Ditemukan {len(files)} file di direktori {source_name}.")

    count = 0
    for file in files:
        try:
            # Membaca id_string dan full_text saja
            df_temp = pd.read_csv(file, usecols=['id_string', 'full_text'], dtype={'id_string': str})

            # Cleaning spasi
            df_temp['id_string'] = df_temp['id_string'].str.strip()
            df_temp['full_text'] = df_temp['full_text'].astype(str).str.strip()

            batch_dfs.append(df_temp)
            count += len(df_temp)
        except Exception as e:
            print(f"⚠️ Gagal membaca {os.path.basename(file)}: {e}")

    print(f"Total baris yang diekstrak dari {source_name}: {count}")

# Eksekusi ekstraksi
extract_data_from_files(seed_dir, "Seed Batches")
extract_data_from_files(refill_dir, "Refill Batches")

# Menggabungkan semua data batch MENTAH (Duplikat TETAP DIPERTAHANKAN di tahap ini)
if batch_dfs:
    target_df_raw = pd.concat(batch_dfs, ignore_index=True)

    # Mengganti nama kolom agar tidak bentrok (collision) dengan kolom dari file Master saat di-merge
    target_df_raw = target_df_raw.rename(columns={
        'id_string': 'batch_id',
        'full_text': 'batch_text'
    })

    print(f"\n✅ Total Data Mentah (Termasuk Duplikat) yang siap dipetakan: {len(target_df_raw)}")
else:
    target_df_raw = pd.DataFrame(columns=['batch_id', 'batch_text'])
    print("❌ Tidak ada data yang berhasil diekstrak dari batch.")

Mulai mengumpulkan data dari file pecahan...
Ditemukan 12 file di direktori Seed Batches.
Total baris yang diekstrak dari Seed Batches: 2400
Ditemukan 24 file di direktori Refill Batches.
Total baris yang diekstrak dari Refill Batches: 4440

✅ Total Data Mentah (Termasuk Duplikat) yang siap dipetakan: 6840


In [14]:
# ==========================================
# 4. Load Master, Mapping & Audit Keyword
# ==========================================
print("Sedang memuat data master... (Ini mungkin memakan waktu jika file besar)")

master_df = pd.read_csv(
    master_path,
    dtype={'id_str': str},
    encoding_errors='ignore',
    low_memory=False
)
print(f"Data master berhasil dimuat. Total baris: {len(master_df)}")

# Cleaning kolom kunci master
master_df['id_str'] = master_df['id_str'].astype(str).str.strip()
master_df['full_text_clean'] = master_df['full_text'].astype(str).str.strip() if 'full_text' in master_df.columns else ""

# ==========================================
# TAHAP 1: Mapping Berdasarkan ID
# ==========================================
# Merge data batch dengan master berdasarkan ID (Inner Join)
# Ini akan menduplikasi baris master_df jika ID di batch muncul lebih dari 1 kali (Sesuai keinginan)
matched_id_df = pd.merge(target_df_raw, master_df, left_on='batch_id', right_on='id_str', how='inner')
print(f"✅ Mapping Tahap 1: {len(matched_id_df)} baris mentah berhasil dipetakan via ID.")

# ==========================================
# TAHAP 2: Mapping Berdasarkan Teks (Fallback)
# ==========================================
# Mencari sisa baris batch mentah yang gagal terpetakan by ID
matched_batch_ids = set(matched_id_df['batch_id'])
unmatched_target_df = target_df_raw[~target_df_raw['batch_id'].isin(matched_batch_ids)]

matched_text_df = pd.DataFrame()
if len(unmatched_target_df) > 0:
    print(f"⚠️ Mencoba mapping {len(unmatched_target_df)} sisa data via Fallback Teks...")

    # Ambil sisa data master yang belum terambil di Tahap 1 (Drop duplikat teks di master agar tidak meledak)
    master_fallback = master_df[~master_df['id_str'].isin(set(matched_id_df['id_str']))]
    master_fallback = master_fallback.drop_duplicates(subset=['full_text_clean'])

    # Merge sisa batch dengan fallback master berdasarkan Teks
    matched_text_df = pd.merge(unmatched_target_df, master_fallback, left_on='batch_text', right_on='full_text_clean', how='inner')
    print(f"✅ Mapping Tahap 2: {len(matched_text_df)} baris mentah berhasil dipetakan via Teks.")

# Gabungkan hasil mapping (Data masih memuat duplikat dari batch)
all_matched_raw = pd.concat([matched_id_df, matched_text_df], ignore_index=True)

# ==========================================
# AUDIT KEYWORD DARI MASTER (SEBELUM DEDUPLIKASI)
# ==========================================
total_raw_matched = len(all_matched_raw)
count_mbg, count_mbg_gratis = 0, 0

if 'source_keyword' in all_matched_raw.columns:
    kw = all_matched_raw['source_keyword'].astype(str).str.strip().str.lower()
    count_mbg = (kw == 'mbg').sum()
    count_mbg_gratis = (kw == 'makan_bergizi_gratis').sum()

count_unknown = total_raw_matched - count_mbg - count_mbg_gratis

pct_mbg = (count_mbg / total_raw_matched) * 100 if total_raw_matched > 0 else 0
pct_mbg_gratis = (count_mbg_gratis / total_raw_matched) * 100 if total_raw_matched > 0 else 0
pct_unknown = (count_unknown / total_raw_matched) * 100 if total_raw_matched > 0 else 0

print("\n📊 AUDIT DISTRIBUSI KEYWORD (Berdasarkan Data Mentah & Duplikat):")
print(f"Total Baris Mentah yang Cocok: {total_raw_matched}")
print(f"- MBG                  : {count_mbg} ({pct_mbg:.1f}%)")
print(f"- Makan Bergizi Gratis : {count_mbg_gratis} ({pct_mbg_gratis:.1f}%)")
print(f"- Tidak Ada Label/Lain : {count_unknown} ({pct_unknown:.1f}%)")

Sedang memuat data master... (Ini mungkin memakan waktu jika file besar)
Data master berhasil dimuat. Total baris: 158712
✅ Mapping Tahap 1: 6840 baris mentah berhasil dipetakan via ID.

📊 AUDIT DISTRIBUSI KEYWORD (Berdasarkan Data Mentah & Duplikat):
Total Baris Mentah yang Cocok: 6840
- MBG                  : 3603 (52.7%)
- Makan Bergizi Gratis : 3237 (47.3%)
- Tidak Ada Label/Lain : 0 (0.0%)


In [15]:
# ==========================================
# 5. Deduplikasi Akhir & Menyimpan Hasil
# ==========================================
print("\nMelakukan deduplikasi pada data yang telah dipetakan...")

# Menghapus duplikasi untuk membuat final dataset mbg_sampled.csv
# Berdasarkan id_str dari master_df agar pasti unik
sampled_df = all_matched_raw.drop_duplicates(subset=['id_str']).copy()

# Membuang kolom sementara yang tadi dipakai untuk join ('batch_id', 'batch_text', 'full_text_clean')
sampled_df = sampled_df.drop(columns=['batch_id', 'batch_text', 'full_text_clean'], errors='ignore')

# Menyimpan dataframe final tanpa index
print(f"Menyimpan data sampel akhir ke:\n{output_path}")
sampled_df.to_csv(output_path, index=False)

# ==========================================
# SUMMARY AKHIR
# ==========================================
total_batch_raw = len(target_df_raw)
final_missing = total_batch_raw - len(all_matched_raw)

print("\n🎉 Proses Pembuatan Sampled Data Selesai!")
print("================== Summary ==================")
print(f"1. Total Baris Mentah Batch          : {total_batch_raw}")
print(f"2. Total Baris Mentah Terpetakan     : {len(all_matched_raw)}")
print(f"3. Total Baris Gagal Terpetakan      : {final_missing if final_missing > 0 else 0}")
print(f"---------------------------------------------")
print(f"4. TOTAL DATA AKHIR (Setelah Dedup)  : {len(sampled_df)}")
print(f"5. Total kolom di output data        : {len(sampled_df.columns)} (Tetap utuh dari master)")
print("=============================================")

# Tampilkan 3 sampel teratas untuk memastikan data aman
sampled_df.head(3)


Melakukan deduplikasi pada data yang telah dipetakan...
Menyimpan data sampel akhir ke:
/content/drive/My Drive/skripsi/dataset/mbg_2025/processed/mbg_sampled_research.csv

🎉 Proses Pembuatan Sampled Data Selesai!
================== Summary ==================
1. Total Baris Mentah Batch          : 6840
2. Total Baris Mentah Terpetakan     : 6840
3. Total Baris Gagal Terpetakan      : 0
---------------------------------------------
4. TOTAL DATA AKHIR (Setelah Dedup)  : 6809
5. Total kolom di output data        : 16 (Tetap utuh dari master)


,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username,source_keyword
0,1975116559027950018,2025-10-06T08:30:58.000Z,0,Setiap anak di Papua berhak tumbuh sehat. Deng...,1975116559027950018,https://pbs.twimg.com/media/G2kGAuDbsAAt6jU.jpg,NaN,in,NaN,0,0,0,https://x.com/sambatdsk/status/197511655902795...,1622465303144988672,sambatdsk,mbg
1,1975164757779898372,2025-10-06T11:42:29.000Z,120,Platform MBG Watch segera launching. Yuk kawal...,1975164757779898372,https://pbs.twimg.com/media/G2kx4TMbMAAXyBA.jpg,NaN,in,NaN,4,3,43,https://x.com/salam4jari/status/19751647577798...,1752301333263392769,salam4jari,mbg
2,1974419204016410757,2025-10-06T14:28:01.000Z,0,@nirwanarasa plis kalo kamu mau yg gratisan tu...,1975206414319751366,NaN,nirwanarasa,in,NaN,0,1,0,https://x.com/rendehuang/status/19752064143197...,1062595086,rendehuang,mbg
